In [ ]:
suppressPackageStartupMessages({
  library(tradeSeq)
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(ggplot2)
  library(matrixStats)
  library(cluster)
  library(RColorBrewer)
  library(SingleCellExperiment)
  library(slingshot)
  library(Matrix)
  library(BiocParallel)
  library(patchwork)
})

RNGversion("4.3.0")
set.seed(1)

# This notebook is intentionally self-contained with respect to the older
# DimensionalityReductionClustering notebooks. It reads primary tradeSeq inputs
# and writes regenerated outputs to SELF_CONTAINED_OUTDIR.
workspace_notebook_dir <- normalizePath(getwd(), mustWork = TRUE)
original_path_file <- file.path(workspace_notebook_dir, "OriginalDirectoryPath.txt")
if (!file.exists(original_path_file)) {
  stop("Run this notebook from k562-tnfa-puromycin.hg38/, where OriginalDirectoryPath.txt is available.")
}

original_notebook_override <- Sys.getenv("CHRONOPACK_ORIGINAL_NOTEBOOK_DIR", "")
if (nzchar(original_notebook_override)) {
  puro_notebook_dir <- normalizePath(path.expand(original_notebook_override), mustWork = TRUE)
} else {
  puro_notebook_dir <- normalizePath(path.expand(readLines(original_path_file, warn = FALSE)[1]), mustWork = TRUE)
}
puro_root <- normalizePath(file.path(puro_notebook_dir, ".."), mustWork = TRUE)
analysis_root <- normalizePath(file.path(puro_root, ".."), mustWork = TRUE)
root_k562a <- file.path(analysis_root, "k562-tnfa.hg38.rep")
root_lps <- file.path(analysis_root, "thp1-k562-lps.hg38.rep")
root_puro <- puro_root

SELF_CONTAINED_OUTDIR <- Sys.getenv(
  "CHRONOPACK_DIMRED_OUTDIR",
  file.path(workspace_notebook_dir, "self_contained_outputs", "global_analysis_extended")
)
dir.create(SELF_CONTAINED_OUTDIR, recursive = TRUE, showWarnings = FALSE)
SELF_CONTAINED_OUTDIR <- normalizePath(SELF_CONTAINED_OUTDIR, mustWork = TRUE)

source(file.path(workspace_notebook_dir, "..", "chronopack", "R", "dimred_clustering_helpers.R"))
source(file.path(workspace_notebook_dir, "..", "chronopack", "R", "dimred_plotting_helpers.R"))

# Keep subsequent saves local to the regenerated output directory.
setwd(SELF_CONTAINED_OUTDIR)
SELF_CONTAINED_OUTDIR


## Plotting Helpers

Reusable dimensionality-reduction plotting helpers are loaded from `chronopack/R/dimred_plotting_helpers.R`.


# 1. Self-Contained Data Build


In [ ]:
# Dataset configuration. DimensionalityReduction&Clustering consumes only
# fixed tradeSeq outputs. The fixed PrepareForTradeSeq/TradeSeqFits pipeline is
# responsible for removing low-count and time-tag-bias-dominated genes and for
# fitting any cross-context extension genes.
cfg <- chronopack_dimred_config(
  root_puro = root_puro,
  root_k562a = root_k562a,
  root_lps = root_lps
)

loaded <- load_dimred_fixed_tradeSeq_outputs(cfg)
list2env(loaded, envir = .GlobalEnv)

names(cfg)


# 2. Construct Trajectory Matrix


In [ ]:
options(repr.plot.width = 4.8, repr.plot.height = 3.6, repr.plot.res = 300)

# Real-time alignment used before constructing the shared trajectory matrix.
time_map <- c(K562a = 110, K562_puro = 110, THP1 = 165, K562b = 165)
offset_map <- c(K562a = 0, K562_puro = 0, THP1 = -10, K562b = -68)

list(time_map = time_map, offset_map = offset_map)


In [ ]:
keys_dimred <- c("K562a", "K562_puro", "K562b", "THP1")
feature_info <- select_dimred_features(
  asso_df.ls = asso_df.ls,
  gamList.ls = gamList.ls,
  keys = keys_dimred,
  pvalue_threshold = 0.05
)

features.ls <- feature_info$features.ls
avail_genes.ls <- feature_info$avail_genes.ls
features <- feature_info$features
avail_genes <- feature_info$avail_genes

c(
  n_union_significant_features = length(features),
  n_common_available_genes = length(avail_genes),
  n_usable_features = length(feature_info$usable_features)
)


In [ ]:
t_interval_map <- c(K562a = 10, K562_puro = 10, K562b = 15, THP1 = 15)

traj.ls <- extract_dimred_smooth_trajectories(
  gamList.ls = gamList.ls,
  sce.ls = sce.ls,
  sf.ls = sf.ls,
  PT.ls = PT.ls,
  features = features,
  keys = keys_dimred,
  t_interval_map = t_interval_map,
  lineage_pick = 1
)

lapply(traj.ls, function(x) dim(x$traj_mat))


In [ ]:
align_res <- align_rescale_dimred_trajectories(
  traj.ls = traj.ls,
  time_map = time_map,
  offset_map = offset_map,
  scale_map = if (exists("scale_map")) scale_map else NULL,
  default_scale_overrides = c(THP1 = 0.65)
)

traj_trunc_rescaled.ls <- align_res$traj_trunc_rescaled.ls
scale_map_use <- align_res$scale_map_use
shared_minutes <- align_res$shared_minutes
keys_use <- align_res$keys_use

traj_trunc_rescaled.ls


In [ ]:
traj_res <- construct_dimred_traj_matrix(traj_trunc_rescaled.ls)
tt <- traj_res$tt
traj_mat <- traj_res$traj_mat

saveRDS(traj_mat, "traj_mat.rds")
saveRDS(tt, "tt.rds")

c(n_genes = nrow(traj_mat), n_timepoints = ncol(traj_mat))


# 3. Dimensionality Reduction And Filtering


In [ ]:
set.seed(1)

pc_df <- build_dimred_pc_df(
  traj_mat = traj_mat,
  tt = tt,
  asso_df.ls = asso_df.ls,
  scale_map_use = scale_map_use,
  descriptor_args = list(
    N_minutes = 10 / length(tt),
    z_diff_threshold = 0.2,
    require_consecutive = 30,
    onset_mode = "horizontal",
    pseudocount = 0
  )
)

dim(pc_df)


In [ ]:
cat("Remaining NA counts:
")
print(colSums(is.na(pc_df[, c("waldStat_1", "meanLogFC", "pvalue_1", "qval_1")])))
print(head(pc_df[, c("gene", "waldStat_1", "meanLogFC", "pvalue_1", "qval_1")], 10))


In [ ]:
matrix_res <- prepare_dimred_matrices(
  traj_mat = traj_mat,
  tt = tt,
  pc_df = pc_df,
  significance_col = "pvalue_1",
  significance_threshold = 0.05
)

X <- matrix_res$X
Xd <- matrix_res$Xd
X_mat <- matrix_res$X_mat
Xd_mat <- matrix_res$Xd_mat
after_filter_note <- "PCA/clustering use normalized trajectories and derivatives, not PC-score distances."

stopifnot(identical(rownames(X_mat), rownames(Xd_mat)))


In [ ]:
dim(X)
dim(Xd)
dim(X_mat)
dim(Xd_mat)


### Remove Low Significance Genes


Fixed-input filtering provenance

Low-count and time-tag-bias-dominated genes are no longer removed in this notebook. The fixed `PrepareForTradeSeq` outputs are expected to have already applied those filters, and the fixed `TradeSeqFits` notebooks should fit only that corrected gene universe. Consequently, dimensionality reduction and clustering start from the fixed tradeSeq outputs directly.


# 4. PCA


In [ ]:
pca_res <- compute_dimred_pca(X_mat, Xd_mat, variance_threshold = 0.99)

pca_X <- pca_res$pca_X
pca_Xd <- pca_res$pca_Xd
X_pc <- pca_res$X_pc
Xd_pc <- pca_res$Xd_pc

pca_res$n_pc_X


In [ ]:
pca_res$n_pc_Xd


In [ ]:
options(repr.plot.width = 4.8, repr.plot.height = 3.6, repr.plot.res = 300)

load_df_long <- data.frame(time = tt,
                           PC1 = pca_X$rotation[, 1],
                           PC2 = pca_X$rotation[, 2],
                           PC3 = pca_X$rotation[, 3],
                           PC4 = pca_X$rotation[, 4]) %>%
                pivot_longer(-time, names_to = "PC", values_to = "loading")


ggplot(load_df_long, aes(x = time, y = loading, color = PC)) + geom_line(linewidth = 1.2) + theme_classic()

PCA reconstruction plotting is provided by `plot_pca_reconstruction()` from `chronopack/R/dimred_plotting_helpers.R`.


In [ ]:
pc_df <- add_dimred_pca_columns(pc_df, pca_X, n_columns = 4)
head(pc_df[, c("gene", "cell_type", "PC1", "PC2", "PC3", "PC4")])


# 5. Clustering Helpers


Clustering metric helpers are loaded from `chronopack/R/dimred_clustering_helpers.R`.


Cluster evaluation helpers are loaded from `chronopack/R/dimred_clustering_helpers.R`.


In [ ]:
# Use direct trajectory and derivative matrices for clustering.
# Each block is scaled by total column variance before alpha weighting.
# PCA scores are retained for PC1-PC4 visualization only.
clustering_inputs <- prepare_dimred_clustering_inputs(X_mat, Xd_mat)
X_n <- clustering_inputs$X_n
Xd_n <- clustering_inputs$Xd_n

cl.ls <- list()
sil_df.ls <- list()
D.ls <- list()


# 6. Finalized Clustering


In [ ]:
lt <- 0
alpha <- 0.25

induced_res <- run_dimred_signed_clustering(
  pc_df = pc_df,
  X_n = X_n,
  Xd_n = Xd_n,
  sign = "induced",
  alpha = alpha,
  log2fc_threshold = lt,
  K_grid = 2:100,
  wss_fraction = 0.95,
  select_k_by = "elbow"
)

induced_res$clust_eval$selection
options(repr.plot.width = 7.5, repr.plot.height = 5)
plot_trajectory_clustering_metrics(induced_res$clust_eval, overlay = FALSE)


Induced-cluster distance construction is handled by `run_dimred_signed_clustering()`.


Induced-cluster evaluation is handled by `run_dimred_signed_clustering()`.


Induced broad and fine cluster assignments are added below in a shared assignment step.


In [ ]:
repressed_res <- run_dimred_signed_clustering(
  pc_df = pc_df,
  X_n = X_n,
  Xd_n = Xd_n,
  sign = "repressed",
  alpha = alpha,
  log2fc_threshold = lt,
  K_grid = 2:100,
  wss_fraction = 0.95,
  select_k_by = "elbow"
)

repressed_res$clust_eval$selection
options(repr.plot.width = 7.5, repr.plot.height = 5)
plot_trajectory_clustering_metrics(repressed_res$clust_eval, overlay = FALSE)


In [ ]:
pc_df <- add_dimred_signed_clusters(pc_df, induced_res, repressed_res)

cl.ls$broad_induced <- induced_res$broad
cl.ls$fine_induced <- induced_res$fine
cl.ls$broad_repressed <- repressed_res$broad
cl.ls$fine_repressed <- repressed_res$fine

sil_df.ls$broad_induced <- induced_res$broad_df
sil_df.ls$fine_induced <- induced_res$fine_df
sil_df.ls$broad_repressed <- repressed_res$broad_df
sil_df.ls$fine_repressed <- repressed_res$fine_df

D.ls$induced <- induced_res$D_use
D.ls$repressed <- repressed_res$D_use

for (name in names(cl.ls)) {
  print(name)
  print(sort(table(cl.ls[[name]]), decreasing = TRUE))
}


Broad induced cluster labels are renamed by `finalize_dimred_pc_df()`.


Broad repressed cluster labels are renamed by `finalize_dimred_pc_df()`.


In [ ]:
pc_df <- finalize_dimred_pc_df(
  pc_df,
  cell_type_order = c("K562a", "K562b", "K562_puro", "THP1")
)

table(pc_df$broad_cluster)


Cluster distance coalescing and final row ordering are handled by `finalize_dimred_pc_df()`.


# 7. Save Regenerated pc_df


In [ ]:
write.csv(pc_df, file.path(SELF_CONTAINED_OUTDIR, "pc_df.new.csv"))
saveRDS(pc_df, file.path(SELF_CONTAINED_OUTDIR, "pc_df.new.rds"))

cat("Wrote:\n")
cat("  ", file.path(SELF_CONTAINED_OUTDIR, "pc_df.new.csv"), "\n", sep = "")
cat("  ", file.path(SELF_CONTAINED_OUTDIR, "pc_df.new.rds"), "\n", sep = "")


# 8. Optional Reference Validation


In [ ]:
VALIDATE_AGAINST_REFERENCE <- identical(Sys.getenv("CHRONOPACK_VALIDATE_REFERENCE"), "1")
if (VALIDATE_AGAINST_REFERENCE) {
  reference_path <- file.path(root_puro, "intmd", "global_analysis", "extended", "pc_df.new.rds")
  ref <- readRDS(reference_path)
  common_rows <- intersect(rownames(ref), rownames(pc_df))
  common_cols <- intersect(colnames(ref), colnames(pc_df))
  cat("Reference rows:", nrow(ref), " regenerated rows:", nrow(pc_df), " common:", length(common_rows), "\n")
  cat("Common columns:", length(common_cols), "\n")
  for (nm in intersect(c("broad_cluster", "cluster_broad_induced", "cluster_broad_repressed", "cluster_fine_induced", "cluster_fine_repressed"), common_cols)) {
    cat(nm, "match fraction:", mean(as.character(ref[common_rows, nm]) == as.character(pc_df[common_rows, nm]), na.rm = TRUE), "\n")
  }
  numeric_cols <- common_cols[vapply(pc_df[common_cols], is.numeric, logical(1)) & vapply(ref[common_cols], is.numeric, logical(1))]
  max_abs <- vapply(numeric_cols, function(nm) max(abs(ref[common_rows, nm] - pc_df[common_rows, nm]), na.rm = TRUE), numeric(1))
  print(sort(max_abs, decreasing = TRUE))
}
